# Optimized Interface Deployment

- `TGI` excels in enterprise-level deployment with its production-ready features. It comes with built-in Kubernetes support and includes everything you need for running in production, like monitoring through Prometheus and Grafana, automatic scaling, and comprehensive safety features. The system also includes enterprise-grade logging and various protective measures like content filtering and rate limiting to keep your deployment secure and stable.

- `vLLM` takes a more flexible, developer-friendly approach to deployment. It’s built with Python at its core and can easily replace OpenAI’s API in your existing applications. The framework focuses on delivering raw performance and can be customized to fit your specific needs. It works particularly well with Ray for managing clusters, making it a great choice when you need high performance and adaptability.

- `llama.cpp` prioritizes simplicity and portability. Its server implementation is lightweight and can run on a wide range of hardware, from powerful servers to consumer laptops and even some high-end mobile devices. With minimal dependencies and a simple C/C++ core, it’s easy to deploy in environments where installing Python frameworks would be challenging. The server provides an OpenAI-compatible API while maintaining a much smaller resource footprint than other solutions.

## Installatoin and basic setup 

`TGI` 

```python
docker run --gpus all \
    --shm-size 1g \
    -p 8080:80 \
    -v ~/.cache/huggingface:/data \
    ghcr.io/huggingface/text-generation-inference:latest \
    --model-id HuggingFaceTB/SmolLM2-360M-Instruct
```

In [ ]:
from huggingface_hub import InferenceClient


client = InferenceClient(
    model="http://localhost:8080"
)

response = client.text_generation(
    "Tell me a story",
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.95,
    details=True,
    stop_sequences=[],
)

print(response.generated_text)

In [ ]:
# for chat format
response = client.chat_completion(
    messages=[
        {"role":"system", "content":"you are helpful assistant."},
        {"role":"user", "content":"Tell me a story."}
    ],
    max_tokens=100,
    temperature=0.9,
    top_p=0.95
)

print(response.choices)

In [ ]:
print(response.choices[0])

In [ ]:
print(response.choices[0].message)

In [ ]:
print(response.choices[0].message.content)

In [ ]:
# Alternatively, you can use the OpenAI client:




# from openai import OpenAI

# # Initialize client pointing to TGI endpoint
# client = OpenAI(
#     base_url="http://localhost:8080/v1",  # Make sure to include /v1
#     api_key="not-needed",  # TGI doesn't require an API key by default
# )

# # Chat completion
# response = client.chat.completions.create(
#     model="HuggingFaceTB/SmolLM2-360M-Instruct",
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "Tell me a story"},
#     ],
#     max_tokens=100,
#     temperature=0.7,
#     top_p=0.95,
# )
# print(response.choices[0].message.content)

## Installing llmama.cpp

### Clone the repository
`git clone https://github.com/ggerganov/llama.cpp`
`cd llama.cpp`

### Build the project
`make`

### Download the SmolLM2-1.7B-Instruct-GGUF model
`curl -L -O https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF/resolve/main/smollm2-1.7b-instruct.Q4_K_M.gguf`

### Start the server
```python
./server \
    -m smollm2-1.7b-instruct.Q4_K_M.gguf \
    --host 0.0.0.0 \
    --port 8080 \
    -c 4096 \
    --n-gpu-layers 0  # Set to a higher number to use GPU
```

In [ ]:
from huggingface_hub import InferenceClient

# Initialize client pointing to llama.cpp server
client = InferenceClient(
    model="http://localhost:8080/v1",  # URL to the llama.cpp server
    token="sk-no-key-required",  # llama.cpp server requires this placeholder
)

# Text generation
response = client.text_generation(
    "Tell me a story",
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.95,
    details=True,
)
print(response.generated_text)

# For chat format
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)

In [ ]:
from openai import OpenAI

# Initialize client pointing to llama.cpp server
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="sk-no-key-required",  # llama.cpp server requires this placeholder
)

# Chat completion
response = client.chat.completions.create(
    model="smollm2-1.7b-instruct",  # Model identifier can be anything as server only loads one model
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)

## Installing vLLM

`uv pip install vllm`
or
`pip install vllm`

``` python
python -m vllm.entrypoints.openai.api_server \
    --model HuggingFaceTB/SmolLM2-360M-Instruct \
    --host 0.0.0.0 \
    --port 8000
```

## Installing using docker

```python
docker run --rm --gpus all nvidia/cuda:12.6.3-runtime-ubuntu24.04 nvidia-smi
```

```python
docker pull vllm/vllm-openai:latest
```

`https://hub.docker.com/r/vllm/vllm-openai`

## Run the server

```python
docker run --rm \
    --runtime=nvidia \
    --gpus all \
    -p 8000:8000 \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    vllm/vllm-openai:latest \
    --model HuggingFaceTB/SmolLM2-360M-Instruct \
    --gpu-memory-utilization 0.90 \
    --max-model-len 2048
```

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    model = "http://localhost:8000/v1",
)


# getting response from the model
response = client.text_generation(
    "can you tell me a story?",
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.95,
    details=True,
)

print(response.generated_text)

# For chat format
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)


## Using openai

`uv pip install openai` or `pip install openai`

In [ ]:
from openai import OpenAI

# Initialize client pointing to vLLM endpoint
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",  # vLLM doesn't require an API key by default
)

# Chat completion
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)